# 01 - Data inspection

Inspects the four original CSV files (read-only). The full written report is `reports/data_quality_report.md`, produced by `scripts/inspect_data.py`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
from climate_yield.config import load_config
cfg = load_config()

In [2]:
from climate_yield import data_loader as dl, data_validation as dv
raw = dl.load_raw_tables(cfg)
for name, df in raw.items():
    rep = dv.validate_raw(name, df)
    print(f'{name:12s} {df.shape}  errors={rep.errors}  warnings={rep.warnings}')

yield        (63691, 7)  errors=[]  warnings=['5 zero yields (treated as missing: log undefined, implausible)']
pesticides   (4789, 7)  errors=[]  warnings=[]
rainfall     (272832, 5)  errors=[]  warnings=[]
temperature  (272832, 5)  errors=[]  warnings=[]


In [3]:
for name, df in raw.items():
    print('=' * 20, name)
    print(df.dtypes.to_string())
    display(df.head(3))

==================== yield
Domain     object
Area       object
Element    object
Item       object
Year        int64
Unit       object
Value       int64


,Domain,Area,Element,Item,Year,Unit,Value
0,Crops,Afghanistan,Yield,Barley,1961,hg/ha,10800
1,Crops,Afghanistan,Yield,Barley,1962,hg/ha,10800
2,Crops,Afghanistan,Yield,Barley,1963,hg/ha,10800


==================== pesticides
Domain     object
Area       object
Element    object
Item       object
Year        int64
Unit       object
Value       int64


,Domain,Area,Element,Item,Year,Unit,Value
0,Pesticides Use,Albania,Agricultural Use,Pesticides (total),1990,tonnes,121
1,Pesticides Use,Albania,Agricultural Use,Pesticides (total),1991,tonnes,121
2,Pesticides Use,Albania,Agricultural Use,Pesticides (total),1992,tonnes,121


==================== rainfall
Rainfall - (MM)    float64
Year                 int64
Statistics          object
Country             object
ISO3                object


,Rainfall - (MM),Year,Statistics,Country,ISO3
0,55.7448,1901,Jan Average,Afghanistan,AFG
1,13.9480,1901,Feb Average,Afghanistan,AFG
2,43.8464,1901,Mar Average,Afghanistan,AFG


==================== temperature
Temperature - (Celsius)    float64
Year                         int64
Statistics                  object
Country                     object
ISO3                        object


,Temperature - (Celsius),Year,Statistics,Country,ISO3
0,-0.03780,1901,Jan Average,Afghanistan,AFG
1,2.42348,1901,Feb Average,Afghanistan,AFG
2,9.36230,1901,Mar Average,Afghanistan,AFG


## Yield: target variable

`Element` is always *Yield* and `Unit` always *hg/ha*, so `Value` is yield per harvested hectare (not production).

In [4]:
y = raw['yield']
print(y[['Domain', 'Element', 'Unit']].drop_duplicates())
y.groupby('Item').agg(rows=('Value', 'size'), countries=('Area', 'nunique'), median=('Value', 'median'))

  Domain Element   Unit
0  Crops   Yield  hg/ha


,rows,countries,median
Item,,,
Barley,5214,114,20092.5
Buckwheat,1186,36,9558.0
Carrots and turnips,5805,138,186177.0
Cauliflowers and broccoli,4413,100,143909.0
Garlic,4473,109,61684.0
Ginger,1484,40,62536.0
Hops,1589,41,13529.0
Maize,8966,178,17835.0
Oats,3773,84,17114.0


In [5]:
tidy = dl.load_tidy_tables(cfg, raw)
runs = dv.constant_runs(tidy['yield'], 5)
jumps = dv.extreme_jumps(tidy['yield'], 5)
print(len(runs), 'flat runs;', len(jumps), 'extreme jumps')
jumps.sort_values('ratio').head()

198 flat runs; 88 extreme jumps


,area,crop,year,yield_hg_ha,previous,ratio
11,Cabo Verde,Maize,2017,1.0,1780.0,0.000562
84,Trinidad and Tobago,Ginger,2018,200.0,13333.0,0.015000
79,Tajikistan,Buckwheat,1998,205.0,5510.0,0.037205
25,Cyprus,Wheat,1973,482.0,10126.0,0.047600
36,Iran (Islamic Republic of),Rye,2008,696.0,13842.0,0.050282


## Climate and pesticides

In [6]:
print(raw['rainfall']['Statistics'].unique())
print(raw['pesticides'][['Element', 'Item', 'Unit']].drop_duplicates())

['Jan Average' 'Feb Average' 'Mar Average' 'Apr Average' 'May Average'
 'Jun Average' 'Jul Average' 'Aug Average' 'Sep Average' 'Oct Average'
 'Nov Average' 'Dec Average']
            Element                Item    Unit
0  Agricultural Use  Pesticides (total)  tonnes


## Integration

In [7]:
from climate_yield.pipeline import prepare_data
data = prepare_data(cfg, save=False)
for k, v in data.integration_report.items():
    print(f'{k}: {v}')

yield_rows_raw: 63691
pesticide_rows_raw: 4789
climate_area_years: 22736
excluded_aggregate_areas: {'areas': ['China'], 'rows': 802}
zero_yield_rows_removed: 5
areas_without_climate: ['Belgium-Luxembourg', 'China, Hong Kong SAR', 'China, Taiwan Province of', 'Cook Islands', 'Czechoslovakia', 'Ethiopia PDR', 'French Guyana', 'French Polynesia', 'Guadeloupe', 'Martinique', 'Niue', 'Palestine', 'Réunion', 'Serbia and Montenegro', 'Sudan (former)', 'USSR', 'Yugoslav SFR']
rows_without_climate_area: 3714
climate_areas_without_yield: ['Andorra', 'Greenland', 'Kiribati', 'Liechtenstein', 'Marshall Islands', 'Monaco', 'Northern Mariana Islands', 'Palau', 'Samoa', 'Seychelles', 'Tuvalu']
pesticide_areas_without_yield: ['Bermuda', 'China, Macao SAR', 'Samoa', 'Seychelles']
yield_areas_without_pesticides: ['Afghanistan', 'Benin', 'Bosnia and Herzegovina', 'Cambodia', 'Cuba', "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Djibouti', 'Dominica', 'Equatorial Guinea', '